# Publiser variabeldefinisjon eksternt

Denne Notebook er for deg som:

- Har en kvalitetsjekket variabeldefinisjon med status `UTKAST` eller `PUBLISERT_INTERNT`
- Ønsker å publisere eksternt

Forutsetninger:
- Du må ha `Kortnavn` for variabeldefinisjonen du vil publisere


## Oppsett

Koden under kjøres som forberedelse for påfølgende steg

In [ ]:
# Nødvendig import
import logging
import sys

from dapla_metadata.variable_definitions import Vardef
from dapla_metadata.variable_definitions import models

# Redusere størrelsen på Traceback for mer tydelige feilmeldinger
%xmode Minimal

# Gjøre at logging vises
logging.basicConfig(
    format="%(levelname)s: %(message)s",
    level=logging.INFO,
    stream=sys.stdout,
    force=True,
)

## Viktig informasjon

Merk at prosessen med å publisere er irreversibel. Når en variabeldefinisjon er publisert eksternt kan den ikke slettes, avpubliseres eller endres tilbake til å kun være publisert internt.

⚠️ Ved ekstern publisering må alle obligatoriske felt ha verdi.

🍀 Flerspråklige felt (som for eksempel `navn` eller `definisjon`) må ha verdier på alle språk (bokmål, nynorsk og engelsk). 

📋 Lister (som for eksempel `enhetstyper`) må ha minst et element i listen.


## Publiser en variabel eksternt

### Hent variabelen

1. Kjør cellen under. 
2. Vent til input-feltet vises
3. Skriv inn kortnavn 
4. Trykk **Enter**.

⚠️  Viktig: Trykk alltid **Enter**, selv om du ikke skriver noe, for å unngå at Jupyter-kjernen henger seg opp.

In [ ]:
mitt_kortnavn = input("Skriv inn kortnavn: ").strip()
min_variabel = Vardef.get_variable_definition_by_shortname(short_name=mitt_kortnavn)

if min_variabel.variable_status == models.VariableStatus.PUBLISHED_EXTERNAL:
    print(
        f"Variabeldefinisjon '{min_variabel.short_name}' er allerede publisert eksternt!"
    )
else:
    print(
        f"Variabeldefinisjon valgt: {min_variabel.short_name} (ID: {min_variabel.id}). Klar for ekstern publisering."
    )

### Publiser den eksternt

✅ I koden under lagres oppdatert status i Vardef for variabelen hentet i steget over. 



In [ ]:
min_variabel = min_variabel.publish_external()
print(f"Ny status: {min_variabel.variable_status}")

## Publiser mange eksternt

### Hent variablene du ønsker å publisere eksternt

Koden under kan brukes til å publisere flere variabler eksternt. Legg inn kortnavnene du vil publisere i listen under.

In [ ]:
# Husk komma mellom kortnavnene, f.eks.: ["kortnavn1", "kortnavn2", "kortnavn3"]
min_kortnavn_liste = [""]
print(f"✅ {len(min_kortnavn_liste)} vil bli publisert eksternt")

### Publiser dem

Kjør koden under for å publisere variablene hentet i steget over.

In [ ]:
publiserte = []
ikke_funnet = []
publisert_tidligere = []
annen_feil = []
for mitt_kortnavn in min_kortnavn_liste:
    try:
        min_variabel = Vardef.get_variable_definition_by_shortname(
            short_name=mitt_kortnavn
        )
        min_variabel = min_variabel.publish_external()
        publiserte.append(mitt_kortnavn)
    except Exception as e:
        if "not found" in str(e):
            ikke_funnet.append(mitt_kortnavn)
        elif "is already published" in str(e):
            publisert_tidligere.append(mitt_kortnavn)
        else:
            print(f"⚠️ Ukjent feil for id {mitt_kortnavn}: {e}")
            annen_feil.append((mitt_kortnavn, str(e)))


print(f"\n📊 Resultatoppsummering:")
print(f" - Antall publisert: {len(publiserte)}")
print(f" - Antall ikke funnet: {len(ikke_funnet)}")
print(f" - Antall allerede publisert: {len(publisert_tidligere)}")
print(f" - Antall med annen feil: {len(annen_feil)}\n")


### Detaljert oversikt

Koden under gir en detaljert oversikt over hva som skjedde med hvert kortnavn der publisering ikke gikk bra

In [ ]:
print("\nIkke fullført publisering:")

print("\n❌ Variabeldefinisjon eksisterer ikke:")
if ikke_funnet:
    for p in ikke_funnet:
        print(f" - {p}")
else:
    print(" - Ingen")

print("\n⚠️ Allerede migrert:")
if publisert_tidligere:
    for p in publisert_tidligere:
        print(f" - {p}")
else:
    print(" - Ingen")

print("\n🚨 Øvrige feil:")
if annen_feil:
    for p, feilmelding in annen_feil:
        print(f" - id {p}: {feilmelding}")
else:
    print(" - Ingen")